## Data Cleaning and Merging Objective

This notebook loads the raw FRED CSV files, verifies the structure of each dataset, standardizes date columns, renames variables, checks missing values, and prepares the data for weekly alignment and feature engineering.

In [5]:
# Upload data files

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

RAW_DATA_PATH = Path(
    r"C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Stress Indicators\Raw Data"
)

stlfsi4 = pd.read_csv(RAW_DATA_PATH / "STLFSI4.csv")
dgs2 = pd.read_csv(RAW_DATA_PATH / "DGS2.csv")
vixcls = pd.read_csv(RAW_DATA_PATH / "VIXCLS.csv")
dgs10 = pd.read_csv(RAW_DATA_PATH / "DGS10.csv")
t10y2y = pd.read_csv(RAW_DATA_PATH / "T10Y2Y.csv")
baa10y = pd.read_csv(RAW_DATA_PATH / "BAA10Y.csv")
sp500 = pd.read_csv(RAW_DATA_PATH / "SP500.csv")

datasets = {
    "STLFSI4": stlfsi4,
    "DGS2": dgs2,
    "VIXCLS": vixcls,
    "DGS10": dgs10,
    "T10Y2Y": t10y2y,
    "BAA10Y": baa10y,
    "SP500": sp500
}

def display_side_by_side(dfs, title, n=5, per_row=7):
    print(title)
    
    items = list(dfs.items())
    
    for i in range(0, len(items), per_row):
        row_items = items[i:i + per_row]
        
        html = "<div style='display:flex; gap:20px; align-items:flex-start; margin-bottom:20px;'>"
        
        for name, df in row_items:
            html += f"""
            <div>
                <h4 style='margin-bottom:5px;'>{name}</h4>
                {df.head(n).to_html(index=False) if title == "Heads:" else df.tail(n).to_html(index=False)}
            </div>
            """
        
        html += "</div>"
        display(HTML(html))

display_side_by_side(datasets, "Heads:", n=5, per_row=7)
display_side_by_side(datasets, "Tails:", n=5, per_row=7)

shapes = pd.DataFrame({
    "Dataset": list(datasets.keys()),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

print("Shapes:")
display(shapes)

Heads:


observation_date,STLFSI4
2016-04-29,0.0685
2016-05-06,0.2686
2016-05-13,0.1787
2016-05-20,0.0768
2016-05-27,-0.0029
observation_date,DGS2
2016-05-02,0.80
2016-05-03,0.75
2016-05-04,0.75
2016-05-05,0.72


Tails:


observation_date,STLFSI4
2026-03-27,-0.1804
2026-04-03,-0.2404
2026-04-10,-0.6472
2026-04-17,-0.7580
2026-04-24,-0.6782
observation_date,DGS2
2026-04-24,3.78
2026-04-27,3.78
2026-04-28,3.84
2026-04-29,3.92


Shapes:


,Dataset,Rows,Columns
0,STLFSI4,522,2
1,DGS2,2609,2
2,VIXCLS,2609,2
3,DGS10,2609,2
4,T10Y2Y,2610,2
5,BAA10Y,2609,2
6,SP500,2610,2


# Data Cleaning and Preprocessing Plan

This analysis uses seven financial time-series datasets downloaded from FRED: the St. Louis Fed Financial Stress Index (`STLFSI4`), VIX, 2-year Treasury yield, 10-year Treasury yield, the 10Y–2Y Treasury spread, the Baa corporate credit spread, and the S&P 500 Index.

The first cleaning step is to standardize each dataset so that every file has a common date column and one clearly named financial variable. The date column will be converted into datetime format, and each value column will be converted into numeric format.

Because `STLFSI4` is a weekly financial stress index while the other market indicators are daily series, the daily predictors will later be converted to weekly frequency. The S&P 500 price series will be transformed into returns before being used as a predictor.

After the datasets are standardized, they will be merged by date, missing values will be reviewed, and lagged predictors will be created. The final modeling dataset will use information from week `t` to forecast financial stress changes in week `t+1`, which avoids look-ahead bias.

In [6]:
# Check raw data structure and missing values

datasets = {
    "STLFSI4": stlfsi4,
    "DGS2": dgs2,
    "VIXCLS": vixcls,
    "DGS10": dgs10,
    "T10Y2Y": t10y2y,
    "BAA10Y": baa10y,
    "SP500": sp500
}

def clean_fred_series(df, new_value_name):
    df = df.copy()
    
    date_col = df.columns[0]
    value_col = df.columns[1]
    
    df = df.rename(columns={
        date_col: "date",
        value_col: new_value_name
    })
    
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[new_value_name] = pd.to_numeric(df[new_value_name], errors="coerce")
    
    df = df.sort_values("date").drop_duplicates(subset="date", keep="last").reset_index(drop=True)
    
    return df

stress = clean_fred_series(stlfsi4, "stress")
dgs2_clean = clean_fred_series(dgs2, "dgs2")
vix_clean = clean_fred_series(vixcls, "vix")
dgs10_clean = clean_fred_series(dgs10, "dgs10")
yield_curve_clean = clean_fred_series(t10y2y, "yield_curve")
credit_spread_clean = clean_fred_series(baa10y, "credit_spread")
sp500_clean = clean_fred_series(sp500, "sp500")

cleaned_datasets = {
    "stress": stress,
    "dgs2": dgs2_clean,
    "vix": vix_clean,
    "dgs10": dgs10_clean,
    "yield_curve": yield_curve_clean,
    "credit_spread": credit_spread_clean,
    "sp500": sp500_clean
}

summary = []

for name, df in cleaned_datasets.items():
    value_col = df.columns[1]
    
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "start_date": df["date"].min(),
        "end_date": df["date"].max(),
        "missing_values": df[value_col].isna().sum(),
        "duplicate_dates": df["date"].duplicated().sum(),
        "value_dtype": df[value_col].dtype
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,dataset,rows,start_date,end_date,missing_values,duplicate_dates,value_dtype
0,stress,522,2016-04-29,2026-04-24,0,0,float64
1,dgs2,2609,2016-05-02,2026-04-30,109,0,float64
2,vix,2609,2016-05-02,2026-04-30,66,0,float64
3,dgs10,2609,2016-05-02,2026-04-30,109,0,float64
4,yield_curve,2610,2016-05-02,2026-05-01,109,0,float64
5,credit_spread,2609,2016-05-02,2026-04-30,114,0,float64
6,sp500,2610,2016-05-02,2026-05-01,95,0,float64


In [7]:
# Weekly align and merge datasets

def to_weekly_last(df, value_col):
    weekly = (
        df.set_index("date")[[value_col]]
          .resample("W-FRI")
          .last()
          .reset_index()
    )
    return weekly

# STLFSI4 is already weekly, but this keeps it aligned to Friday weeks
stress_weekly = to_weekly_last(stress, "stress")

# Convert daily predictors to weekly Friday values
vix_weekly = to_weekly_last(vix_clean, "vix")
dgs2_weekly = to_weekly_last(dgs2_clean, "dgs2")
dgs10_weekly = to_weekly_last(dgs10_clean, "dgs10")
yield_curve_weekly = to_weekly_last(yield_curve_clean, "yield_curve")
credit_spread_weekly = to_weekly_last(credit_spread_clean, "credit_spread")

# Convert S&P 500 prices into weekly log returns
sp500_temp = sp500_clean.copy()
sp500_temp["sp500_log_return_daily"] = np.log(sp500_temp["sp500"] / sp500_temp["sp500"].shift(1))

sp500_weekly = (
    sp500_temp.set_index("date")[["sp500_log_return_daily"]]
    .resample("W-FRI")
    .sum(min_count=1)
    .rename(columns={"sp500_log_return_daily": "sp500_return"})
    .reset_index()
)

# Merge everything onto the weekly stress index
weekly_data = (
    stress_weekly
    .merge(vix_weekly, on="date", how="left")
    .merge(dgs2_weekly, on="date", how="left")
    .merge(dgs10_weekly, on="date", how="left")
    .merge(yield_curve_weekly, on="date", how="left")
    .merge(credit_spread_weekly, on="date", how="left")
    .merge(sp500_weekly, on="date", how="left")
)

print("Weekly merged data:")
display(weekly_data.head())
display(weekly_data.tail())

print("Shape:")
print(weekly_data.shape)

print("Missing values after weekly merge:")
display(weekly_data.isna().sum())

Weekly merged data:


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return
0,2016-04-29,0.0685,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-05-06,0.2686,14.72,0.74,1.79,1.05,2.87,-0.011738
2,2016-05-13,0.1787,15.04,0.76,1.71,0.95,2.90,-0.005132
3,2016-05-20,0.0768,15.20,0.89,1.85,0.96,2.86,0.002786
4,2016-05-27,-0.0029,13.12,0.90,1.85,0.95,2.85,0.022519


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return
517,2026-03-27,-0.1804,31.05,3.88,4.44,0.56,1.78,-0.021380
518,2026-04-03,-0.2404,23.87,3.84,4.35,0.51,1.74,0.033025
519,2026-04-10,-0.6472,19.23,3.81,4.31,0.50,1.72,0.030543
520,2026-04-17,-0.7580,17.48,3.71,4.26,0.55,1.72,0.044355
521,2026-04-24,-0.6782,18.71,3.78,4.31,0.53,1.70,0.005461


Shape:
(522, 8)
Missing values after weekly merge:


date             0
stress           0
vix              1
dgs2             1
dgs10            1
yield_curve      1
credit_spread    1
sp500_return     1
dtype: int64

Only issue is the first week has missing predictors because STLFSI4 starts on April 29th of 2016, while the rest of the files fill in on the week following. Therefore the first row cannot be used for modeling and will be dropped. 

In [8]:
# Create forecasting targets and remove incomplete rows

model_data = weekly_data.copy()

# Forecast next week's stress level
model_data["stress_next"] = model_data["stress"].shift(-1)

# Forecast next week's change in stress
model_data["stress_change_next"] = model_data["stress_next"] - model_data["stress"]

# Remove rows with missing predictors or missing future target
model_data = model_data.dropna().reset_index(drop=True)

print("Model data:")
display(model_data.head())
display(model_data.tail())

print("Shape:")
print(model_data.shape)

print("Missing values:")
display(model_data.isna().sum())

Model data:


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next
0,2016-05-06,0.2686,14.72,0.74,1.79,1.05,2.87,-0.011738,0.1787,-0.0899
1,2016-05-13,0.1787,15.04,0.76,1.71,0.95,2.90,-0.005132,0.0768,-0.1019
2,2016-05-20,0.0768,15.20,0.89,1.85,0.96,2.86,0.002786,-0.0029,-0.0797
3,2016-05-27,-0.0029,13.12,0.90,1.85,0.95,2.85,0.022519,0.1254,0.1283
4,2016-06-03,0.1254,13.47,0.78,1.71,0.93,2.86,0.001034,0.1522,0.0268


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next
515,2026-03-20,-0.3658,26.78,3.88,4.39,0.51,1.79,-0.019136,-0.1804,0.1854
516,2026-03-27,-0.1804,31.05,3.88,4.44,0.56,1.78,-0.021380,-0.2404,-0.0600
517,2026-04-03,-0.2404,23.87,3.84,4.35,0.51,1.74,0.033025,-0.6472,-0.4068
518,2026-04-10,-0.6472,19.23,3.81,4.31,0.50,1.72,0.030543,-0.7580,-0.1108
519,2026-04-17,-0.7580,17.48,3.71,4.26,0.55,1.72,0.044355,-0.6782,0.0798


Shape:
(520, 10)
Missing values:


date                  0
stress                0
vix                   0
dgs2                  0
dgs10                 0
yield_curve           0
credit_spread         0
sp500_return          0
stress_next           0
stress_change_next    0
dtype: int64

Looks clear, we can move to the next step which is the lag feature creation where making the data valid for forecasting by using past information to predict next-week stress changes. 

In [9]:
# Create lagged predictors

lag_data = model_data.copy()

predictor_cols = [
    "stress",
    "vix",
    "dgs2",
    "dgs10",
    "yield_curve",
    "credit_spread",
    "sp500_return"
]

for col in predictor_cols:
    lag_data[f"{col}_lag1"] = lag_data[col].shift(1)
    lag_data[f"{col}_lag4"] = lag_data[col].shift(4)

lag_data["sp500_return_4w_mean"] = lag_data["sp500_return"].shift(1).rolling(window=4).mean()
lag_data["sp500_abs_return_4w_mean"] = lag_data["sp500_return"].abs().shift(1).rolling(window=4).mean()

lag_data = lag_data.dropna().reset_index(drop=True)

print("Lagged modeling data:")
display(lag_data.head())
display(lag_data.tail())

print("Shape:")
print(lag_data.shape)

print("Missing values:")
display(lag_data.isna().sum())

Lagged modeling data:


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next,stress_lag1,stress_lag4,vix_lag1,vix_lag4,dgs2_lag1,dgs2_lag4,dgs10_lag1,dgs10_lag4,yield_curve_lag1,yield_curve_lag4,credit_spread_lag1,credit_spread_lag4,sp500_return_lag1,sp500_return_lag4,sp500_return_4w_mean,sp500_abs_return_4w_mean
0,2016-06-03,0.1254,13.47,0.78,1.71,0.93,2.86,0.001034,0.1522,0.0268,-0.0029,0.2686,13.12,14.72,0.90,0.74,1.85,1.79,0.95,1.05,2.85,2.87,0.022519,-0.011738,0.002109,0.010544
1,2016-06-10,0.1522,17.03,0.73,1.64,0.91,2.86,-0.001459,0.5669,0.4147,0.1254,0.1787,13.47,15.04,0.78,0.76,1.71,1.71,0.93,0.95,2.86,2.90,0.001034,-0.005132,0.005302,0.007868
2,2016-06-17,0.5669,19.41,0.70,1.62,0.92,2.89,-0.011926,0.2572,-0.3097,0.1522,0.0768,17.03,15.20,0.73,0.89,1.64,1.85,0.91,0.96,2.86,2.86,-0.001459,0.002786,0.006220,0.006949
3,2016-06-24,0.2572,25.76,0.64,1.57,0.93,2.98,-0.016458,0.5807,0.3235,0.5669,-0.0029,19.41,13.12,0.70,0.90,1.62,1.85,0.92,0.95,2.89,2.85,-0.011926,0.022519,0.002542,0.009235
4,2016-07-01,0.5807,14.77,0.59,1.46,0.87,2.86,0.031662,0.2051,-0.3756,0.2572,0.1254,25.76,13.47,0.64,0.78,1.57,1.71,0.93,0.93,2.98,2.86,-0.016458,0.001034,-0.007202,0.007719


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next,stress_lag1,stress_lag4,vix_lag1,vix_lag4,dgs2_lag1,dgs2_lag4,dgs10_lag1,dgs10_lag4,yield_curve_lag1,yield_curve_lag4,credit_spread_lag1,credit_spread_lag4,sp500_return_lag1,sp500_return_lag4,sp500_return_4w_mean,sp500_abs_return_4w_mean
511,2026-03-20,-0.3658,26.78,3.88,4.39,0.51,1.79,-0.019136,-0.1804,0.1854,-0.2972,-0.4875,27.19,19.09,3.73,3.48,4.28,4.08,0.55,0.60,1.83,1.69,-0.016128,0.009640,-0.007831,0.012651
512,2026-03-27,-0.1804,31.05,3.88,4.44,0.56,1.78,-0.021380,-0.2404,-0.0600,-0.3658,-0.4423,26.78,19.86,3.88,3.38,4.39,3.97,0.51,0.59,1.79,1.80,-0.019136,-0.004443,-0.015025,0.015025
513,2026-04-03,-0.2404,23.87,3.84,4.35,0.51,1.74,0.033025,-0.6472,-0.4068,-0.1804,-0.4272,31.05,29.49,3.88,3.56,4.44,4.15,0.56,0.59,1.78,1.75,-0.021380,-0.020393,-0.019259,0.019259
514,2026-04-10,-0.6472,19.23,3.81,4.31,0.50,1.72,0.030543,-0.7580,-0.1108,-0.2404,-0.2972,23.87,27.19,3.84,3.73,4.35,4.28,0.51,0.55,1.74,1.83,0.033025,-0.016128,-0.005905,0.022417
515,2026-04-17,-0.7580,17.48,3.71,4.26,0.55,1.72,0.044355,-0.6782,0.0798,-0.6472,-0.3658,19.23,26.78,3.81,3.88,4.31,4.39,0.50,0.51,1.72,1.79,0.030543,-0.019136,0.005763,0.026021


Shape:
(516, 26)
Missing values:


date                        0
stress                      0
vix                         0
dgs2                        0
dgs10                       0
yield_curve                 0
credit_spread               0
sp500_return                0
stress_next                 0
stress_change_next          0
stress_lag1                 0
stress_lag4                 0
vix_lag1                    0
vix_lag4                    0
dgs2_lag1                   0
dgs2_lag4                   0
dgs10_lag1                  0
dgs10_lag4                  0
yield_curve_lag1            0
yield_curve_lag4            0
credit_spread_lag1          0
credit_spread_lag4          0
sp500_return_lag1           0
sp500_return_lag4           0
sp500_return_4w_mean        0
sp500_abs_return_4w_mean    0
dtype: int64

## Final Clean notebook check

In [10]:
print("Final cleaned dataset shape:")
print(lag_data.shape)

print("\nDate range:")
print(lag_data["date"].min(), "to", lag_data["date"].max())

print("\nDuplicate dates:")
print(lag_data["date"].duplicated().sum())

print("\nMissing values:")
print(lag_data.isna().sum().sum())

print("\nColumns:")
print(lag_data.columns.tolist())

display(lag_data.head())
display(lag_data.tail())

Final cleaned dataset shape:
(516, 26)

Date range:
2016-06-03 00:00:00 to 2026-04-17 00:00:00

Duplicate dates:
0

Missing values:
0

Columns:
['date', 'stress', 'vix', 'dgs2', 'dgs10', 'yield_curve', 'credit_spread', 'sp500_return', 'stress_next', 'stress_change_next', 'stress_lag1', 'stress_lag4', 'vix_lag1', 'vix_lag4', 'dgs2_lag1', 'dgs2_lag4', 'dgs10_lag1', 'dgs10_lag4', 'yield_curve_lag1', 'yield_curve_lag4', 'credit_spread_lag1', 'credit_spread_lag4', 'sp500_return_lag1', 'sp500_return_lag4', 'sp500_return_4w_mean', 'sp500_abs_return_4w_mean']


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next,stress_lag1,stress_lag4,vix_lag1,vix_lag4,dgs2_lag1,dgs2_lag4,dgs10_lag1,dgs10_lag4,yield_curve_lag1,yield_curve_lag4,credit_spread_lag1,credit_spread_lag4,sp500_return_lag1,sp500_return_lag4,sp500_return_4w_mean,sp500_abs_return_4w_mean
0,2016-06-03,0.1254,13.47,0.78,1.71,0.93,2.86,0.001034,0.1522,0.0268,-0.0029,0.2686,13.12,14.72,0.90,0.74,1.85,1.79,0.95,1.05,2.85,2.87,0.022519,-0.011738,0.002109,0.010544
1,2016-06-10,0.1522,17.03,0.73,1.64,0.91,2.86,-0.001459,0.5669,0.4147,0.1254,0.1787,13.47,15.04,0.78,0.76,1.71,1.71,0.93,0.95,2.86,2.90,0.001034,-0.005132,0.005302,0.007868
2,2016-06-17,0.5669,19.41,0.70,1.62,0.92,2.89,-0.011926,0.2572,-0.3097,0.1522,0.0768,17.03,15.20,0.73,0.89,1.64,1.85,0.91,0.96,2.86,2.86,-0.001459,0.002786,0.006220,0.006949
3,2016-06-24,0.2572,25.76,0.64,1.57,0.93,2.98,-0.016458,0.5807,0.3235,0.5669,-0.0029,19.41,13.12,0.70,0.90,1.62,1.85,0.92,0.95,2.89,2.85,-0.011926,0.022519,0.002542,0.009235
4,2016-07-01,0.5807,14.77,0.59,1.46,0.87,2.86,0.031662,0.2051,-0.3756,0.2572,0.1254,25.76,13.47,0.64,0.78,1.57,1.71,0.93,0.93,2.98,2.86,-0.016458,0.001034,-0.007202,0.007719


,date,stress,vix,dgs2,dgs10,yield_curve,credit_spread,sp500_return,stress_next,stress_change_next,stress_lag1,stress_lag4,vix_lag1,vix_lag4,dgs2_lag1,dgs2_lag4,dgs10_lag1,dgs10_lag4,yield_curve_lag1,yield_curve_lag4,credit_spread_lag1,credit_spread_lag4,sp500_return_lag1,sp500_return_lag4,sp500_return_4w_mean,sp500_abs_return_4w_mean
511,2026-03-20,-0.3658,26.78,3.88,4.39,0.51,1.79,-0.019136,-0.1804,0.1854,-0.2972,-0.4875,27.19,19.09,3.73,3.48,4.28,4.08,0.55,0.60,1.83,1.69,-0.016128,0.009640,-0.007831,0.012651
512,2026-03-27,-0.1804,31.05,3.88,4.44,0.56,1.78,-0.021380,-0.2404,-0.0600,-0.3658,-0.4423,26.78,19.86,3.88,3.38,4.39,3.97,0.51,0.59,1.79,1.80,-0.019136,-0.004443,-0.015025,0.015025
513,2026-04-03,-0.2404,23.87,3.84,4.35,0.51,1.74,0.033025,-0.6472,-0.4068,-0.1804,-0.4272,31.05,29.49,3.88,3.56,4.44,4.15,0.56,0.59,1.78,1.75,-0.021380,-0.020393,-0.019259,0.019259
514,2026-04-10,-0.6472,19.23,3.81,4.31,0.50,1.72,0.030543,-0.7580,-0.1108,-0.2404,-0.2972,23.87,27.19,3.84,3.73,4.35,4.28,0.51,0.55,1.74,1.83,0.033025,-0.016128,-0.005905,0.022417
515,2026-04-17,-0.7580,17.48,3.71,4.26,0.55,1.72,0.044355,-0.6782,0.0798,-0.6472,-0.3658,19.23,26.78,3.81,3.88,4.31,4.39,0.50,0.51,1.72,1.79,0.030543,-0.019136,0.005763,0.026021


In [11]:
# Save final cleaned modeling dataset

OUTPUT_PATH = Path(
    r"C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Stress Indicators"
)

lag_data.to_csv(OUTPUT_PATH / "fsi_model_data.csv", index=False)

print("Saved cleaned modeling dataset to:")
print(OUTPUT_PATH / "fsi_model_data.csv")

Saved cleaned modeling dataset to:
C:\Users\DiBip\Downloads\Georgia Tech\Time Series Analysis\Final Project\Financial Stress Indicators\fsi_model_data.csv
